# Customer Segmentation & RFM Analysis

## Project Overview

This project analyzes approximately two years of online retail transaction data to identify meaningful customer segments using **Recency, Frequency, and Monetary (RFM) analysis**. The objective is to identify high-value customers, recognize historically valuable customers who have become inactive, and develop segment-specific retention and marketing recommendations.

**Tools:** Python, pandas, Matplotlib, Jupyter Notebook

**Dataset:** Online Retail II, UCI Machine Learning Repository (December 2009–December 2011).

Source: https://archive.ics.uci.edu/dataset/502/online%2Bretail%2Bii  
Citation: Chen, D. (2012). *Online Retail II* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CG6D

### Business Questions
1. What does overall customer and transaction activity look like?
2. Who are the most valuable customers based on RFM behavior?
3. How is the customer base distributed across meaningful RFM segments?
4. Which segments generate the most spending?
5. Which historically valuable customers have become inactive or appear at risk?
6. What distinguishes the major customer segments?
7. What actions should the business take for each important segment?


## 1. Data Loading & Initial Inspection

The workbook contains two yearly worksheets. The data is loaded, inspected for dimensions and data types, and prepared for customer-level analysis. Customer ID is treated as an identifier rather than a numeric measure.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

file_path = "online_retail_II.xlsx"
year_2009_2010 = pd.read_excel(file_path, sheet_name="Year 2009-2010")
year_2010_2011 = pd.read_excel(file_path, sheet_name="Year 2010-2011")

print("2009-2010 shape:", year_2009_2010.shape)
print("2010-2011 shape:", year_2010_2011.shape)
year_2009_2010.info()


## 2. Data Cleaning & Preparation

Customer IDs are converted to strings, missing IDs are quantified, line-level transaction value is created, both years are combined, and cancellations/non-positive transactions are investigated before constructing the RFM base.


In [ ]:
for df in (year_2009_2010, year_2010_2011):
    df["Customer ID"] = df["Customer ID"].astype("Int64").astype("string")
    df["LineTotal"] = df["Quantity"] * df["Price"]

for label, df in [("2009-2010", year_2009_2010), ("2010-2011", year_2010_2011)]:
    missing = df["Customer ID"].isna().sum()
    missing_pct = missing / len(df) * 100
    missing_value = df.loc[df["Customer ID"].isna(), "LineTotal"].sum()
    total_value = df["LineTotal"].sum()
    print(label, "missing Customer ID rows:", missing, f"({missing_pct:.2f}%)")
    print(label, "missing-ID transaction value share:", f"{missing_value / total_value * 100:.2f}%")

transactions = pd.concat([year_2009_2010, year_2010_2011], ignore_index=True)

cancelled_rows = transactions["Invoice"].astype("string").str.startswith("C", na=False)
negative_quantity_rows = transactions["Quantity"] < 0
nonpositive_price_rows = transactions["Price"] <= 0

print("Combined rows:", len(transactions))
print("Cancelled rows:", cancelled_rows.sum())
print("Negative quantity rows:", negative_quantity_rows.sum())
print("Non-positive price rows:", nonpositive_price_rows.sum())

rfm_transactions = transactions.loc[
    transactions["Customer ID"].notna()
    & ~cancelled_rows
    & (transactions["Quantity"] > 0)
    & (transactions["Price"] > 0)
].copy()

print("Clean RFM transaction rows:", len(rfm_transactions))
print("Unique customers:", rfm_transactions["Customer ID"].nunique())
print("Date range:", rfm_transactions["InvoiceDate"].min(), "to", rfm_transactions["InvoiceDate"].max())


## 3. RFM Construction

- **Recency:** days since the customer's most recent purchase.
- **Frequency:** number of unique completed orders.
- **Monetary:** total historical completed-purchase spending.

A snapshot date one day after the final transaction is used for Recency.


In [ ]:
snapshot_date = rfm_transactions["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = (
    rfm_transactions
    .groupby("Customer ID")
    .agg(
        Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
        Frequency=("Invoice", "nunique"),
        Monetary=("LineTotal", "sum")
    )
    .reset_index()
)

print("RFM customer rows:", len(rfm))
rfm[["Recency", "Frequency", "Monetary"]].describe()


## 4. RFM Scoring

Recency and Monetary use quartiles. Frequency uses behavior-based thresholds because **27.6% of customers placed exactly one order**, so a naive quartile split would divide customers with identical purchasing behavior.

- **F1:** 1 order
- **F2:** 2–3 orders
- **F3:** 4–7 orders
- **F4:** 8+ orders


In [ ]:
print("One-order customer share:", f"{(rfm['Frequency'] == 1).mean() * 100:.2f}%")

rfm["R_Score"] = pd.qcut(rfm["Recency"], q=4, labels=[4, 3, 2, 1]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["Monetary"], q=4, labels=[1, 2, 3, 4]).astype(int)
rfm["F_Score"] = pd.cut(
    rfm["Frequency"],
    bins=[0, 1, 3, 7, float("inf")],
    labels=[1, 2, 3, 4]
).astype(int)

print(pd.crosstab(rfm["R_Score"], rfm["F_Score"]))


## 5. Customer Segmentation

Segments are based primarily on **Recency and Frequency** to represent lifecycle behavior. Monetary value is retained separately so financial importance can be evaluated independently.


In [ ]:
def assign_segment(row):
    r, f = row["R_Score"], row["F_Score"]
    if r == 4 and f == 4:
        return "Champions"
    elif r >= 3 and f >= 3:
        return "Loyal Customers"
    elif r == 4 and f == 2:
        return "Potential Loyalists"
    elif r == 4 and f == 1:
        return "New Customers"
    elif r == 3 and f in [1, 2]:
        return "Promising"
    elif r <= 2 and f >= 3:
        return "At Risk"
    elif r == 2 and f <= 2:
        return "Needs Attention"
    else:
        return "Hibernating"

rfm["Segment"] = rfm.apply(assign_segment, axis=1)
rfm["Segment"].value_counts()


## 6. Segment Analysis & Key Findings

The segments are compared by customer count, average behavior, and total historical spending to identify where customer value is concentrated and where re-engagement may be worthwhile.


In [ ]:
segment_summary = (
    rfm.groupby("Segment")
    .agg(
        Customers=("Customer ID", "count"),
        Avg_Recency=("Recency", "mean"),
        Avg_Frequency=("Frequency", "mean"),
        Avg_Monetary=("Monetary", "mean"),
        Total_Monetary=("Monetary", "sum")
    )
    .sort_values("Total_Monetary", ascending=False)
)
segment_summary["Customer_%"] = segment_summary["Customers"] / len(rfm) * 100
segment_summary["Monetary_%"] = segment_summary["Total_Monetary"] / rfm["Monetary"].sum() * 100
segment_summary.round(2)

high_value_at_risk = rfm.loc[
    (rfm["Segment"] == "At Risk") & (rfm["M_Score"] >= 3)
].copy()

print("High-value At Risk customers:", len(high_value_at_risk))
print("Historical spending:", f"${high_value_at_risk['Monetary'].sum():,.2f}")


### 6.1 Customer Share vs. Spending Contribution

**Champions represent only 12.2% of customers but account for 54.6% of historical spending.** Loyal Customers add another 21.3% of spending. Hibernating customers represent 21.9% of customers but only 3.6% of spending.


In [ ]:
plot_data = segment_summary[["Customer_%", "Monetary_%"]].rename(
    columns={"Customer_%": "Customer Share", "Monetary_%": "Spending Share"}
).sort_values("Spending Share", ascending=False)

ax = plot_data.plot(kind="bar", figsize=(12, 6))
ax.set_title("Customer Share vs. Spending Share by RFM Segment")
ax.set_xlabel("Customer Segment")
ax.set_ylabel("Share (%)")
plt.xticks(rotation=45, ha="right")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=3, fontsize=8)
ax.set_ylim(0, 60)
plt.tight_layout()
plt.show()


### 6.2 Inactivity by Customer Segment

Hibernating customers average about **522 days** since their last purchase, compared with roughly **269 days** for At-Risk customers and only **10 days** for Champions.


In [ ]:
recency_by_segment = segment_summary["Avg_Recency"].sort_values(ascending=False)
ax = recency_by_segment.plot(kind="barh", figsize=(10, 6))
ax.set_title("Average Days Since Last Purchase by Customer Segment")
ax.set_xlabel("Average Days Since Last Purchase")
ax.set_ylabel("Customer Segment")
ax.invert_yaxis()
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f days", padding=4)
plt.tight_layout()
plt.show()


### 6.3 High-Value At-Risk Customers

**589 high-value At-Risk customers generated approximately $1.99 million in historical spending.** This is historical value, not predicted future revenue loss.


In [ ]:
top_at_risk = high_value_at_risk.sort_values("Monetary", ascending=False).head(10).sort_values("Monetary")
ax = top_at_risk.plot(x="Customer ID", y="Monetary", kind="barh", figsize=(10, 6), legend=False)
ax.set_title("Top 10 High-Value At-Risk Customers by Historical Spending")
ax.set_xlabel("Historical Spending")
ax.set_ylabel("Customer ID")
for container in ax.containers:
    ax.bar_label(container, labels=[f"${v:,.0f}" for v in container.datavalues], padding=4)
plt.tight_layout()
plt.show()


## 7. Overall Customer & Transaction Activity


In [ ]:
order_totals = rfm_transactions.groupby("Invoice")["LineTotal"].sum()
print("Clean transaction rows:", len(rfm_transactions))
print("Unique customers:", rfm_transactions["Customer ID"].nunique())
print("Unique orders:", rfm_transactions["Invoice"].nunique())
print("Total historical spending:", f"${rfm_transactions['LineTotal'].sum():,.2f}")
print("Average order value:", f"${order_totals.mean():,.2f}")
print("Median order value:", f"${order_totals.median():,.2f}")


## 8. Limitations & Considerations

- Missing Customer IDs cannot support customer-level RFM analysis.
- Cancelled invoices, negative quantities, and non-positive prices are excluded; Monetary therefore represents **gross completed-purchase value**, not net value after returns.
- RFM summarizes historical behavior and does not predict churn, future purchases, or revenue.
- Segment definitions are rule-based and dataset-specific.
- Demographic, acquisition-channel, and campaign-response data are not available.


## 9. Business Recommendations

1. **Protect Champions** with loyalty benefits, exclusivity, and strong relationship management.
2. **Prioritize high-value At-Risk customers** for personalized win-back campaigns, starting with the strongest historical spenders and purchasers.
3. **Strengthen Loyal Customers** with repeat-purchase incentives and personalized recommendations.
4. **Develop Potential Loyalists and New Customers** by encouraging a second or subsequent purchase while engagement is recent.
5. **Use lower-cost testing for Promising and Needs-Attention customers** before committing substantial retention resources.
6. **Limit expensive retention efforts for Hibernating customers**, favoring low-cost automated reactivation.

### Overall Recommendation
Customer value is highly concentrated. Retention resources should prioritize **Champions, Loyal Customers, and historically valuable At-Risk customers** rather than treating every customer equally.


## 10. Conclusion

The analysis transformed more than 800,000 cleaned transaction rows into a customer-level view of **5,878 unique customers**. Champions represent **12.2% of customers but 54.6% of historical spending**, while **589 high-value At-Risk customers** account for approximately **$1.99 million in historical spending**.

RFM segmentation provides a practical framework for moving from broad customer reporting to targeted retention and marketing decisions based on observed purchasing behavior.
